<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Modèle de données SQL

Objectif : à partir de la table `inspections` nettoyée, construire un **panel pont × année** avec les événements de dégradation, plus les tables de dimensions.

- **Partie 0** (fournie) : setup + chargement/nettoyage → table `inspections`
- **Étapes 1 à 7** : c'est toi qui écris le SQL

## 0.1 — Setup

In [ ]:
!pip install -q jupysql duckdb-engine

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nbi'

import duckdb
con = duckdb.connect(f'{DATA_DIR}/nbi.duckdb')

%load_ext sql
%sql con
%config SqlMagic.displaylimit = 20

## 0.2 — Chargement + nettoyage → `inspections`

Deux temps :
1. **Lecture (Python/pandas)** → `inspections_raw` : les 19 colonnes utiles, en texte brut, une année à la fois. On passe par pandas car le lecteur CSV de DuckDB n'arrive pas à lire certains fichiers NBI (fins de ligne mélangées `\n` / `\r\n`, apostrophes dans les textes) ; pandas est plus tolérant.
2. **Nettoyage (SQL)** → `inspections` : typage, conversion lat/lon, notes `'N'` → NULL, `min_rating`, `age`.

| Colonne | Contenu |
|---|---|
| `bridge_id` | `state_code` + `-` + `structure_number` |
| `year` | année du fichier NBI (2016–2025 ; le fichier 2015 téléchargé n'est pas exploitable) |
| `state_code`, `county_code`, `lat`, `lon` | localisation (lat/lon en degrés décimaux, lon négative) |
| `owner_code`, `material_code`, `structure_type_code` | codes bruts (items 22, 43A, 43B) |
| `year_built`, `year_reconstructed` | entiers, NULL si 0/absent |
| `adt`, `pct_truck`, `length_m`, `deck_width_m` | trafic et géométrie |
| `deck`, `super`, `sub`, `culvert` | notes 0–9, `'N'` → NULL |
| `is_culvert` | `culvert IS NOT NULL` |
| `min_rating` | `culvert` si ponceau, sinon `LEAST(deck, super, sub)` |
| `age` | `year - year_built` |

Choix de nettoyage :
- on garde seulement `RECORD_TYPE_005A = '1'` (les lignes de type 2 décrivent la route *sous* le pont et dupliqueraient la structure) ;
- les doublons éventuels ne sont **pas** supprimés ici : tu les chercheras à l'étape 6.

⏱️ Lecture de ~2,5 Go depuis le Drive : compte plusieurs minutes.

In [ ]:
# Lecture des fichiers bruts avec pandas, un fichier (= une année) à la fois.
# pandas tolère les fins de ligne mélangées (\n / \r\n) qui font planter le lecteur CSV de DuckDB.
import glob, os, re
import pandas as pd

COLS = [
    'STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'RECORD_TYPE_005A', 'COUNTY_CODE_003',
    'LAT_016', 'LONG_017', 'OWNER_022', 'STRUCTURE_KIND_043A', 'STRUCTURE_TYPE_043B',
    'YEAR_BUILT_027', 'YEAR_RECONSTRUCTED_106', 'ADT_029', 'PERCENT_ADT_TRUCK_109',
    'STRUCTURE_LEN_MT_049', 'DECK_WIDTH_MT_052',
    'DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060', 'CULVERT_COND_062',
]

con.execute('DROP TABLE IF EXISTS inspections_raw')
for f in sorted(glob.glob(f'{DATA_DIR}/raw/nbi_20*.txt')):
    year = int(re.search(r'nbi_(20\d{2})', f).group(1))
    if year < 2016:          # nbi_2015.txt n'est pas un vrai fichier NBI
        continue
    df = pd.read_csv(
        f, sep=',', quotechar="'", encoding='latin-1',
        dtype=object, keep_default_na=False, na_values=[''],   # tout en texte, vide -> NULL
        usecols=lambda c: c.strip() in COLS,
        on_bad_lines='warn',                                # signale les lignes illisibles au lieu de planter
    )
    df.columns = df.columns.str.strip()
    df = df.reindex(columns=COLS)                           # colonne absente une année -> NULL
    df.insert(0, 'year', year)
    if con.sql("SELECT COUNT(*) FROM information_schema.tables WHERE table_name = 'inspections_raw'").fetchone()[0] == 0:
        con.execute('CREATE TABLE inspections_raw AS SELECT * FROM df')
    else:
        con.execute('INSERT INTO inspections_raw SELECT * FROM df')
    print(f'{year} : {len(df):>8,} lignes')
    del df

print(con.sql('SELECT COUNT(*) AS total FROM inspections_raw').fetchone()[0], 'lignes au total')

In [ ]:
%%sql
-- Nettoyage + typage : inspections_raw (texte brut) -> inspections (table propre)
CREATE OR REPLACE TABLE inspections AS
WITH typed AS (
    SELECT
        lpad(trim(STATE_CODE_001), 2, '0') || '-' || trim(STRUCTURE_NUMBER_008) AS bridge_id,
        CAST(year AS INTEGER) AS year,
        lpad(trim(STATE_CODE_001), 2, '0')  AS state_code,
        lpad(trim(COUNTY_CODE_003), 3, '0') AS county_code,

        -- LAT_016 = DDMMSSss, LONG_017 = DDDMMSSss (ouest, donc négatif)
        lpad(trim(LAT_016), 8, '0')  AS lat_txt,
        lpad(trim(LONG_017), 9, '0') AS lon_txt,

        lpad(trim(OWNER_022), 2, '0')            AS owner_code,
        trim(STRUCTURE_KIND_043A)                AS material_code,
        lpad(trim(STRUCTURE_TYPE_043B), 2, '0')  AS structure_type_code,

        NULLIF(TRY_CAST(trim(YEAR_BUILT_027)         AS INTEGER), 0) AS year_built,
        NULLIF(TRY_CAST(trim(YEAR_RECONSTRUCTED_106) AS INTEGER), 0) AS year_reconstructed,

        TRY_CAST(trim(ADT_029)               AS INTEGER) AS adt,
        TRY_CAST(trim(PERCENT_ADT_TRUCK_109) AS INTEGER) AS pct_truck,
        TRY_CAST(trim(STRUCTURE_LEN_MT_049)  AS DOUBLE)  AS length_m,
        TRY_CAST(trim(DECK_WIDTH_MT_052)     AS DOUBLE)  AS deck_width_m,

        -- notes 0-9 ; 'N' (non applicable) ou vide -> NULL
        TRY_CAST(trim(DECK_COND_058)           AS TINYINT) AS deck,
        TRY_CAST(trim(SUPERSTRUCTURE_COND_059) AS TINYINT) AS super,
        TRY_CAST(trim(SUBSTRUCTURE_COND_060)   AS TINYINT) AS sub,
        TRY_CAST(trim(CULVERT_COND_062)        AS TINYINT) AS culvert
    FROM inspections_raw
    WHERE trim(RECORD_TYPE_005A) = '1'   -- 1 = la structure elle-même (2 = route qui passe dessous, doublon)
)
SELECT
    bridge_id, year, state_code, county_code,
    CASE WHEN TRY_CAST(lat_txt AS BIGINT) > 0 THEN
        CAST(substr(lat_txt, 1, 2) AS INTEGER)
      + CAST(substr(lat_txt, 3, 2) AS INTEGER) / 60.0
      + CAST(substr(lat_txt, 5, 4) AS INTEGER) / 100.0 / 3600.0
    END AS lat,
    CASE WHEN TRY_CAST(lon_txt AS BIGINT) > 0 THEN
      -( CAST(substr(lon_txt, 1, 3) AS INTEGER)
       + CAST(substr(lon_txt, 4, 2) AS INTEGER) / 60.0
       + CAST(substr(lon_txt, 6, 4) AS INTEGER) / 100.0 / 3600.0 )
    END AS lon,
    owner_code, material_code, structure_type_code,
    year_built, year_reconstructed,
    adt, pct_truck, length_m, deck_width_m,
    deck, super, sub, culvert,
    culvert IS NOT NULL AS is_culvert,
    CASE WHEN culvert IS NOT NULL THEN culvert
         ELSE LEAST(deck, super, sub)          -- LEAST ignore les NULL dans DuckDB
    END AS min_rating,
    year - year_built AS age
FROM typed;

In [ ]:
%%sql
-- Vérification rapide du chargement (pas l'étape 1 : juste s'assurer que la table existe)
SELECT COUNT(*) AS n_rows, MIN(year) AS first_year, MAX(year) AS last_year
FROM inspections;

In [ ]:
%%sql
SELECT * FROM inspections LIMIT 5;

In [ ]:
%%sql
-- SELECT * FROM (describe inspections) ;
summarize inspections

---
## Étape 1 — Prise en main de la table (15 min)

**Objectif :** répondre à 4 questions, une requête chacune.
1. Combien de lignes (= ponts inspectés) par année ?
2. Combien de ponts **distincts** au total sur toute la période ?
3. Distribution de `min_rating` par année (combien de ponts à 0, 1, …, 9 chaque année).
4. Part de ponceaux (`is_culvert`) — globale ou par année.

**Concepts :** `GROUP BY`, `COUNT(DISTINCT …)`, agrégats conditionnels `COUNT(*) FILTER (WHERE …)`.

**Indices (sans solution) :**
- Q1 et Q2 ne donnent pas le même ordre de grandeur que la somme des lignes : pourquoi ?
- Q3 : deux façons possibles — format « long » (`GROUP BY` sur deux colonnes) ou format « large » (une colonne par note avec `FILTER`). Essaie au moins le format large pour pratiquer `FILTER`.
- Q4 : un booléen se compte avec `FILTER (WHERE is_culvert)`, et une part = compte filtré / compte total. Attention à la division entière.
- Pense aux `NULL` dans `min_rating` : où vont-ils dans tes résultats ?

In [ ]:
%%sql
-- Q1 — nombre de ponts par année
SELECT COUNT (DISTINCT bridge_id) AS nb_bridges_pery FROM inspections
GROUP BY year;

In [ ]:
%%sql
-- Q2 — nombre de ponts distincts
SELECT COUNT (DISTINCT bridge_id) AS distinct_bridges FROM inspections;


In [ ]:
%%sql
-- Q3 — distribution de min_rating par année



In [ ]:
%%sql
-- Q4 — part de ponceaux
